In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

df_clean = pd.read_csv("../../data/processed/df_clean.csv")
cols = ["BALANCE","PURCHASES","ONEOFF_PURCHASES",
        "INSTALLMENTS_PURCHASES","CASH_ADVANCE","CREDIT_LIMIT","PAYMENTS"]
prep = df_clean.copy()
prep[cols] = np.log1p(prep[cols])
scaled = StandardScaler().fit_transform(prep)
X4 = PCA(n_components=4, random_state=0).fit_transform(scaled)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score


In [ ]:
for d in range(2, 6):
    X = X4 if d == 4 else PCA(n_components=d, random_state=0).fit_transform(scaled)
    print(f"dims={d}")
    for k in range(2, 9):
        km = KMeans(n_clusters=k, random_state=0, n_init=10).fit(X)
        sil = silhouette_score(X, km.labels_)
        frac = pd.Series(km.labels_).value_counts(normalize=True).min()
        print(f"  k={k}: silhouette={sil:.3f}  min_cluster={frac:.1%}")


In [ ]:
Xs = {4: X4}
Xs.update({d: PCA(n_components=d, random_state=0).fit_transform(scaled) for d in (2,3,5)})
results = []
for d in (2,3,4,5):
    X = Xs[d]
    for k in range(2, 9):
        km = KMeans(n_clusters=k, random_state=0, n_init=10).fit(X)
        sil = silhouette_score(X, km.labels_)
        frac = pd.Series(km.labels_).value_counts(normalize=True).min()
        results.append((d, k, sil, frac))

df_grid = pd.DataFrame(results, columns=["dims","k","silhouette","min_cluster"])
valid = df_grid[df_grid.min_cluster >= 0.10].sort_values("silhouette", ascending=False)
winner = valid.iloc[0]
print("best valid (dims, k):", int(winner.dims), int(winner.k))
print("silhouette:", round(winner.silhouette, 3), "| min_cluster:", f"{winner.min_cluster:.1%}")


In [ ]:
X2 = PCA(n_components=2, random_state=0).fit_transform(scaled)
for eps in (0.3, 0.5, 0.7, 1.0):
    for ms in (5, 10, 20):
        db = DBSCAN(eps=eps, min_samples=ms).fit(X2)
        n = int(db.labels_.max()) + 1 if set(db.labels_) != {-1} else 0
        noise = (db.labels_ == -1).mean()
        sil = silhouette_score(X2, db.labels_) if n > 1 else float("nan")
        print(f"eps={eps} min_samples={ms:2d} clusters={n:2d} noise={noise:.1%} sil={sil:.3f}")


In [ ]:
db = DBSCAN(eps=0.3, min_samples=10).fit(X2)
n_clusters = int(db.labels_.max()) + 1
noise = (db.labels_ == -1).mean()
db_sil = silhouette_score(X2, db.labels_)

km = KMeans(n_clusters=3, random_state=0, n_init=10).fit(X2)
km_sil = silhouette_score(X2, km.labels_)

print(f"DBSCAN : clusters={n_clusters} noise={noise:.1%} silhouette={db_sil:.3f}")
print(f"KMeans : clusters=3 noise=0.0% silhouette={km_sil:.3f}")
print("verdict: KMeans wins - lower silhouette AND DBSCAN leaves customers unlabelled (noise); FS5 needs a label for every row")


## Grid results: inertia, silhouette and the heatmap


In [ ]:
rows = []
for d in range(2, 6):
    X = PCA(n_components=d, random_state=0).fit_transform(scaled)
    for k in range(2, 9):
        km = KMeans(n_clusters=k, random_state=0, n_init=10).fit(X)
        rows.append({
            "dims": d,
            "k": k,
            "inertia": km.inertia_,
            "silhouette": silhouette_score(X, km.labels_),
            "min_cluster": pd.Series(km.labels_).value_counts(normalize=True).min(),
        })

grid = pd.DataFrame(rows)


In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(grid.pivot(index="dims", columns="k", values="silhouette"),
            annot=True, fmt=".3f", cmap="YlGnBu")
plt.title("Silhouette by PCA components x k")
plt.xlabel("k")
plt.ylabel("PCA components")


## Elbow and silhouette curves (dims=2)


In [ ]:
X2 = PCA(n_components=2, random_state=0).fit_transform(scaled)
curve = grid[grid.dims == 2].sort_values("k")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(curve.k, curve.inertia, "o-")
ax[0].set_title("Elbow - inertia vs k")
ax[1].plot(curve.k, curve.silhouette, "o-")
ax[1].set_title("Silhouette vs k")
for a in ax:
    a.set_xlabel("k")


## DBSCAN grid - best combinations by silhouette


In [ ]:
rows = []
for d in range(2, 6):
    X = PCA(n_components=d, random_state=0).fit_transform(scaled)
    for eps in (0.3, 0.5, 0.7, 1.0):
        for ms in (5, 10, 20):
            db = DBSCAN(eps=eps, min_samples=ms).fit(X)
            n_clusters = len(set(db.labels_)) - 1
            if n_clusters < 2:
                continue
            rows.append({
                "dims": d, "eps": eps, "min_samples": ms,
                "clusters": n_clusters,
                "noise": (db.labels_ == -1).mean(),
                "silhouette": silhouette_score(X, db.labels_),
            })

db_grid = pd.DataFrame(rows).sort_values("silhouette", ascending=False)


## KMeans vs DBSCAN - comparison


In [ ]:
km = KMeans(n_clusters=3, random_state=0, n_init=10).fit(X2)
db = DBSCAN(eps=0.3, min_samples=10).fit(X2)

comparison = pd.DataFrame({
    "method": ["KMeans", "DBSCAN"],
    "silhouette": [silhouette_score(X2, km.labels_), silhouette_score(X2, db.labels_)],
    "davies_bouldin": [davies_bouldin_score(X2, km.labels_), davies_bouldin_score(X2, db.labels_)],
    "calinski_harabasz": [calinski_harabasz_score(X2, km.labels_), calinski_harabasz_score(X2, db.labels_)],
    "noise_pct": [0.0, (db.labels_ == -1).mean()],
})


## Final model - labels attached to df_clean


In [ ]:
df_clean["cluster_kmeans"] = km.labels_
df_clean["cluster_dbscan"] = db.labels_
df_clean["est_attypique_dbscan"] = db.labels_ == -1
df_clean["cluster_final"] = df_clean["cluster_kmeans"]



In [ ]:
share = df_clean.cluster_final.value_counts(normalize=True).sort_index()
print("customers:", len(df_clean))
print("nulls in cluster_final:", df_clean.cluster_final.isna().sum())
print("share per cluster:")


## Final clusters - scatter and centroids


In [ ]:
plt.figure(figsize=(9, 7))
plt.scatter(X2[:, 0], X2[:, 1], c=df_clean["cluster_final"], cmap="tab10", s=8, alpha=0.6)
centroids = pd.DataFrame(km.cluster_centers_, columns=["PC1", "PC2"])
plt.scatter(centroids.PC1, centroids.PC2, marker="X", s=250, c="black", edgecolors="white", linewidths=1.5, label="centroid")
plt.legend()
plt.title("KMeans clusters in PCA space (dims=2, k=3)")
plt.xlabel("PC1")
plt.ylabel("PC2")


## Save the labelled dataset


In [ ]:
df_clean.to_csv("../../data/processed/df_clean_clustered.csv", index=False)
